# 05. Паттерны кликов

Исследуем связь позиции в выдаче с кликами, ценой и брендом.

Прокси CTR: доля кликов бренда на топ-позициях.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_utils import (
    apply_plot_style, save_fig, ensure_dirs, load_query_clicks, load_sku_desc,
    text_len, parquet_num_rows, parquet_schema_names, dataset_overview_stats,
    save_stats, QUERY_CLICKS_PATH, SKU_DESC_PATH, SKUS_PKL_PATH,
    MVIDEO_RED, DARK_SLATE, MUTED,
)
ensure_dirs()
apply_plot_style()
pd.set_option("display.max_colwidth", 80)
print("ROOT:", ROOT)
df = load_query_clicks(n=400_000, seed=42)
print(df.shape)


## Доля кликов по позициям


In [ ]:
pos = df["sku_position"].dropna()
pos = pos[pos <= 40]
vc = pos.value_counts().sort_index(); share = vc / vc.sum(); cum = share.cumsum()
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(share.index, share.values, color=MVIDEO_RED, alpha=0.85, label="Доля кликов")
ax.plot(cum.index, cum.values, color=DARK_SLATE, lw=2, marker="o", ms=3, label="Накопленная доля")
ax.set_title("Доля кликов по позициям и накопленный охват")
ax.set_xlabel("Позиция"); ax.set_ylabel("Доля"); ax.legend()
save_fig(fig, "15_position_click_share.png"); plt.show()
print(f"Доля кликов на позициях 0–2: {share.loc[share.index <= 2].sum():.1%}")


## Цена vs позиция


In [ ]:
sub = df[["sku_price", "sku_position"]].dropna()
sub = sub[(sub["sku_price"] > 0) & (sub["sku_position"] <= 30)]
sub = sub[sub["sku_price"] < sub["sku_price"].quantile(0.95)]
agg = sub.groupby("sku_position")["sku_price"].median().reset_index()
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(agg["sku_position"], agg["sku_price"], color=MVIDEO_RED, lw=2.2, marker="o", ms=4)
ax.fill_between(agg["sku_position"], agg["sku_price"], alpha=0.12, color=MVIDEO_RED)
ax.set_title("Медианная цена vs позиция в выдаче")
ax.set_xlabel("Позиция"); ax.set_ylabel("Медианная цена, ₽")
save_fig(fig, "10_price_vs_position.png"); plt.show()


## Прокси CTR брендов: доля кликов на топ-3 позициях


In [ ]:
bdf = df[df["sku_brand_name"].replace("", np.nan).notna()].copy()
top_brands = bdf["sku_brand_name"].value_counts().head(15).index
bdf = bdf[bdf["sku_brand_name"].isin(top_brands)]
proxy = (bdf.assign(top3=bdf["sku_position"] <= 2).groupby("sku_brand_name")["top3"].mean().sort_values())
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(proxy.index.astype(str), proxy.values, color=MVIDEO_RED)
ax.set_title("Прокси CTR: доля кликов бренда на позициях 0–2")
ax.set_xlabel("Доля кликов на топ-3")
save_fig(fig, "19_brand_top3_share.png"); plt.show()
display(proxy.sort_values(ascending=False).to_frame("top3_share"))


## Средняя позиция по брендам


In [ ]:
avg_pos = bdf.groupby("sku_brand_name")["sku_position"].median().sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(avg_pos.index.astype(str), avg_pos.values, color=DARK_SLATE)
ax.set_title("Медианная позиция клика по топ-брендам"); ax.set_xlabel("Медианная позиция")
save_fig(fig, "20_brand_median_position.png"); plt.show()


## Цена × позиция: hexbin


In [ ]:
plot_df = df[(df["sku_price"] > 0) & (df["sku_position"] <= 25)].copy()
plot_df = plot_df[plot_df["sku_price"] < plot_df["sku_price"].quantile(0.9)]
fig, ax = plt.subplots(figsize=(10, 6))
hb = ax.hexbin(plot_df["sku_position"], plot_df["sku_price"], gridsize=35, cmap="Reds", mincnt=5)
cb = fig.colorbar(hb, ax=ax); cb.set_label("Число кликов")
ax.set_title("Плотность кликов: позиция × цена")
ax.set_xlabel("Позиция"); ax.set_ylabel("Цена, ₽")
save_fig(fig, "21_price_position_hexbin.png"); plt.show()
